# Chapter 1 — The Landscape of Tabular Deep Learning

Reproduces:
- Figure 1.1: Histogram of small-tabular dataset sizes (synthetic illustration).
- Figure 1.2: Tree-vs-DL benchmark slice on synthetic datasets.
- Table 1.1: Tabular FM timeline 2022-2026.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabkernels.benchmarks import run_grinsztajn_slice

_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
np.random.seed(42)

## Figure 1.1: Tabular benchmark dataset sizes

Distribution of dataset sizes typically used in tabular benchmarks (rough characterization).

In [ ]:
# Simulate the size distribution of common tabular benchmark datasets.
# Mostly small (n in [10^3, 10^4]); some medium; few large.
rng = np.random.RandomState(0)
n_datasets = 50
log_sizes = rng.normal(loc=3.5, scale=0.7, size=n_datasets)  # log10 n centered around 3000
log_sizes = np.clip(log_sizes, 2.0, 6.0)
sizes = 10 ** log_sizes

fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
ax.hist(log_sizes, bins=20, color='C0', alpha=0.8, edgecolor='k')
ax.set_xlabel('log10(n)')
ax.set_ylabel('# datasets')
ax.set_title('Figure 1.1: Approximate size distribution of small-tabular benchmark datasets')
ax.axvline(3, ls='--', c='gray', alpha=0.5, label='n=1000')
ax.axvline(4, ls='--', c='gray', alpha=0.5, label='n=10000')
ax.legend()
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_01_01_dataset_sizes.pdf', bbox_inches='tight')
plt.show()

## Figure 1.2: Tree-vs-DL benchmark slice

Compare XGBoost, MLP, and a linear baseline on three synthetic datasets representative of different structure types.

In [ ]:
df = run_grinsztajn_slice(n_train=500, n_test=200, n_seeds=3)
summary = df.groupby(['dataset', 'model'])['test_mse'].agg(['mean', 'std']).reset_index()
print(summary.to_string(index=False))

fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
datasets = summary['dataset'].unique()
models = summary['model'].unique()
x = np.arange(len(datasets))
width = 0.25
for i, model in enumerate(models):
    sub = summary[summary['model'] == model]
    ax.bar(x + (i - 1) * width, sub['mean'].values, width, label=model,
            yerr=sub['std'].values, capsize=4)
ax.set_xticks(x); ax.set_xticklabels(datasets, rotation=15)
ax.set_ylabel('test MSE'); ax.set_title('Figure 1.2: Models on synthetic tabular datasets (3 seeds)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_01_02_grinsztajn_slice.pdf', bbox_inches='tight')
plt.show()

## Table 1.1: Tabular foundation model timeline 2022-2026

In [ ]:
timeline = pd.DataFrame([
    {'Year': 2022, 'Model': 'TabPFN v1', 'Max n': 1000, 'Max p': 100, 'Prior': 'SCM', 'License': 'free'},
    {'Year': 2025, 'Model': 'TabPFNv2',  'Max n': 10000, 'Max p': 500, 'Prior': 'SCM (richer)', 'License': 'free'},
    {'Year': 2025, 'Model': 'TabPFN-2.5','Max n': 50000, 'Max p': 2000, 'Prior': 'SCM', 'License': 'free'},
    {'Year': 2025, 'Model': 'TabICL',    'Max n': 60000, 'Max p': 500, 'Prior': 'SCM', 'License': 'open'},
    {'Year': 2026, 'Model': 'TabICLv2',  'Max n': 100000,'Max p': 1000,'Prior': 'engineered SCM', 'License': 'open'},
    {'Year': 2025, 'Model': 'TARTE',     'Max n': 10000, 'Max p': 200, 'Prior': 'KG-augmented', 'License': 'open'},
    {'Year': 2025, 'Model': 'TabSTAR',   'Max n': 10000, 'Max p': 200, 'Prior': 'verbalised', 'License': 'open'},
])
print(timeline.to_string(index=False))